# Explaining a spoken sentence with an audio language model

**Goal:** let an audio language model hear a recording, then attribute its answer to the spoken words.

**Why it matters:** a spoken instruction is only auditable when you can show which words the model relied on; masking the audio itself, not just the transcript, tests what the model heard.

**Prerequisites:** `uv sync --extra audio --extra transformers`. The first run downloads `ibm-granite/granite-speech-3.3-2b` (about 5.9 GB) and `facebook/wav2vec2-base-960h` (about 360 MB). The clip `voices_sp0307.wav` is from the VOiCES corpus (CC BY 4.0). A GPU or Apple MPS keeps the run to minutes.

**Output:** the model's transcript of the clip, then one attribution value per spoken word.

In [1]:
from pathlib import Path

from mllm_shap import (
    AudioPart,
    ConversationSnapshot,
    ExplainConfig,
    ExplainRunner,
    LmStudioBackend,
    Message,
    Role,
    TextPart,
    TransformersTextBackend,
    Turn,
)
from mllm_shap.plugins import PluginContext

## Hear the clip

A checkpoint that `AutoModelForCausalLM` does not know loads with its processor, so the `transformers` backend takes audio parts. Granite Speech's chat template has no audio slot: the backend puts its `<|audio|>` token before the message text and passes the waveform, resampled to 16 kHz.

In [2]:
MODEL_ID = "ibm-granite/granite-speech-3.3-2b"
REVISION = "4ac2f02f413c6169ae8c0ccc217115a366e552d7"
TRANSCRIPT = "I had that curiosity beside me at this moment"
audio = AudioPart(Path("voices_sp0307.wav"), sample_rate=16000)

settings = {
    "backend": {
        "kind": "transformers",
        "model_id": MODEL_ID,
        # Granite's projector fails under the default attention kernel on Apple MPS.
        "options": {"revision": REVISION, "dtype": "bfloat16", "attn_implementation": "eager"},
    },
    "generation": {"max_new_tokens": 16},
    "explanation": {
        "estimator": "complementary",
        "budget": {"fraction": 0.125},
        "players": "sgpa_segments",
        "absence_policy": "delete",
        "archive": {"path": ""},
        "plugin_options": {"sgpa_segments": {
            "model_id": "facebook/wav2vec2-base-960h",
            "revision": "22aad52d435eb6dbaf354bdad9b0da84ce7d6156",
        }},
    },
}
config = ExplainConfig.model_validate(settings)
backend = TransformersTextBackend(PluginContext(config=config))
ask = Message(
    role=Role.USER,
    content=(audio, TextPart("can you transcribe the speech into a written format?")),
)
heard = await backend.generate(
    ConversationSnapshot.from_turns((Turn(messages=(ask,)),)),
    config.generation.model_copy(update={"max_new_tokens": 48}),
)
print(heard.text)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/953 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


i had that curiosity beside me at this moment


## Attribute the answer to spoken words

With `players: sgpa_segments` each spoken word of the transcript is a player with its stretch of audio. `delete` cuts an absent word from both the transcript and the waveform, so every coalition prompt is a shorter recording. Nine words have 512 coalitions; the complementary estimator samples 64 of them (about 70 s on an Apple M4 (16 GB)), so the values approximate Shapley values and their sum is close to, not equal to, the full answer's score minus the empty one's.

In [3]:
spoken = Message(role=Role.USER, content=(TextPart(TRANSCRIPT), audio))
snapshot = ConversationSnapshot.from_turns((Turn(messages=(spoken,)),))
output = await ExplainRunner(config, backend=backend).explain(snapshot)
result = output.result
for span, value in zip(output.player_set.spans, result.values, strict=True):
    print(f"{span.text:10} {value:+.4f}")
print("efficiency gap:", round(sum(result.values) - (result.grand_value - result.base_value), 3))

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


I          -0.0148
had        +0.1878
that       +0.1149
curiosity  +0.2107
beside     +0.1499
me         +0.1314
at         +0.0770
this       +0.1821
moment     +0.1288
efficiency gap: 0.168


## What the model hears, not what it reads

The run above sends the masked transcript next to the masked audio, so a word can matter because the model reads it. With `transcript: align_only` the transcript only builds the players and every coalition sends the audio alone.

In [4]:
settings["explanation"]["plugin_options"]["sgpa_segments"]["transcript"] = "align_only"
heard_only = await ExplainRunner(ExplainConfig.model_validate(settings), backend=backend).explain(snapshot)
for span, read, heard_value in zip(
    output.player_set.spans, result.values, heard_only.result.values, strict=True
):
    print(f"{span.text:10} send {read:+.4f}   align_only {heard_value:+.4f}")

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


I          send -0.0148   align_only +0.0011
had        send +0.1878   align_only +0.1904
that       send +0.1149   align_only +0.1067
curiosity  send +0.2107   align_only +0.1976
beside     send +0.1499   align_only +0.1011
me         send +0.1314   align_only +0.1374
at         send +0.0770   align_only +0.0920
this       send +0.1821   align_only +0.2223
moment     send +0.1288   align_only +0.1282


## LM Studio and HTTP servers

The LM Studio SDK takes no audio, so the `lmstudio` backend refuses audio parts unless `backend.options.audio` is `transcript`, which sends the text only. The refusal comes before any server call. The `api` backend sends audio as OpenAI `input_audio` parts to servers that accept them.

In [5]:
lms_config = ExplainConfig.model_validate({"backend": {"kind": "lmstudio", "model_id": "any"}})
try:
    await LmStudioBackend(PluginContext(config=lms_config)).generate(snapshot, config.generation)
except ValueError as error:
    print(error)

audio snapshots are not supported by the LM Studio backend (its SDK takes no audio input); set backend.options audio: transcript to score the text parts only
